# Лабораторна робота №2. Частина 1: Попередній аналіз та очищення даних VHI
**Виконав:** Студент групи ФБ-46 — Ільченко Влад

In [2]:
import os
import zipfile
import requests
import pandas as pd
import numpy as np

def download_and_prepare_data():
    """
    Автоматично завантажує та розпаковує датасет, якщо його немає локально,
    після чого зчитує та очищує дані за допомогою Pandas.
    """
    folder_path = "vhi_data"
    zip_name = "vhi_data.zip"
    
    # 1. АВТОЗАВАНТАЖЕННЯ (якщо папки немає)
    if not os.path.exists(folder_path):
        print(" Локальних даних не знайдено. Починаємо автоматичне завантаження...")
        
        # Пряме посилання на архів у твоєму репозиторії
        url = "https://raw.githubusercontent.com/vladilchen-ipt28-source/zpad_2026_labs/main/vhi_data.zip"
        
        try:
            response = requests.get(url, stream=True)
            with open(zip_name, "wb") as f:
                f.write(response.content)
            
            print(" Розпакування архіву...")
            with zipfile.ZipFile(zip_name, "r") as zip_ref:
                zip_ref.extractall(folder_path)
                
            os.remove(zip_name) # Видаляємо тимчасовий zip
            print(" Завантаження та розпакування завершено успішно!")
        except Exception as e:
            print(f" Не вдалося завантажити автоматично: {e}")
            print("Будь ласка, переконайся, що папка 'vhi_data' лежить поруч з цим ноутбуком.")

    # 2. ЗЧИТУВАННЯ ТА РОЗУМНЕ ОЧИЩЕННЯ ДАНИХ
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"Критична помилка: папка '{folder_path}' відсутня.")
        
    all_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
    if not all_files:
        raise FileNotFoundError(f"У папці '{folder_path}' немає .csv файлів.")
        
    combined_data = []
    
    for i, file_name in enumerate(all_files):
        file_path = os.path.join(folder_path, file_name)
        
        # Читаємо файл, ігноруючи поламані рядки
        temp_df = pd.read_csv(file_path, skiprows=1, sep=',', on_bad_lines='skip', index_col=False)
        
        # Базове чищення назв колонок від явних пробілів та ком
        temp_df.columns = [c.replace(' ', '').replace(',', '').strip() for c in temp_df.columns]
        
        # ГНУЧКИЙ ПОШУК КОЛОНОК (Шукаємо частковий збіг імен без врахування регістру)
        rename_dict = {}
        for col in temp_df.columns:
            c_low = col.lower()
            if 'year' in c_low: rename_dict[col] = 'Year'
            elif 'week' in c_low: rename_dict[col] = 'Week'
            elif 'vhi' in c_low or '%' in c_low: rename_dict[col] = 'VHI' # Спіймає і %VHI, і vhi_value
            
        temp_df.rename(columns=rename_dict, inplace=True)
        
        # Перевірка, чи всі 3 критичні колонки успішно розпізнано
        required_cols = ['Year', 'Week', 'VHI']
        if not all(col in temp_df.columns for col in required_cols):
            # Пробуємо прочитати без skiprows, якщо структура зміщена
            temp_df = pd.read_csv(file_path, sep=',', on_bad_lines='skip', index_col=False)
            temp_df.columns = [c.replace(' ', '').replace(',', '').strip() for c in temp_df.columns]
            temp_df.rename(columns=rename_dict, inplace=True)
            
            if not all(col in temp_df.columns for col in required_cols):
                print(f" Пропущено файл {file_name}: не знайдено колонки VHI. Наявні: {list(temp_df.columns)}")
                continue
        
        # Видалення пустих клітинок та фільтрація текстового сміття
        temp_df.dropna(subset=['Year', 'Week', 'VHI'], inplace=True)
        temp_df = temp_df[pd.to_numeric(temp_df['Year'], errors='coerce').notnull()]
        
        # Фінальне приведення типів
        temp_df['Year'] = temp_df['Year'].astype(int)
        temp_df['Week'] = temp_df['Week'].astype(int)
        temp_df['VHI'] = temp_df['VHI'].astype(float)
        
        # Додаємо ID області
        temp_df['Area_ID'] = i + 1
        combined_data.append(temp_df)
        
    if not combined_data:
        raise ValueError("Жоден файл не було успішно зчитано. Перевір формат даних.")
        
    return pd.concat(combined_data, ignore_index=True)

# Запуск конвеєра завантаження та парсингу
df = download_and_prepare_data()
print(f"\n=== ДАНІ УСПІШНО ЗАВАНТАЖЕНО ТА СИНХРОНІЗОВАНО! ===")
print(f"Загальна кількість рядків у DataFrame: {len(df)}")


=== ДАНІ УСПІШНО ЗАВАНТАЖЕНО ТА СИНХРОНІЗОВАНО! ===
Загальна кількість рядків у DataFrame: 60345


### Завдання 1. Вивести екстремуми (мінімум та максимум) індексу VHI для заданої області за конкретний рік.

In [3]:
def get_vhi_extremes(dataframe, area_id, year):
    """Повертає мінімальне та максимальне значення VHI для обраної області та року"""
    filtered = dataframe[(dataframe['Area_ID'] == area_id) & (dataframe['Year'] == year)]
    
    if filtered.empty:
        return None, None
        
    min_vhi = filtered['VHI'].min()
    max_vhi = filtered['VHI'].max()
    return min_vhi, max_vhi

In [4]:
# Тестовий виклик для Області №9 (Київська) за 2020 рік
target_area = 9
target_year = 2020

min_v, max_v = get_vhi_extremes(df, target_area, target_year)

print(f"РЕЗУЛЬТАТ ВИКОНАННЯ ЗАВДАННЯ 1:")
if min_v is not None:
    print(f"Область ID: {target_area}, Рік: {target_year}")
    print(f"-> Мінімальний індекс VHI: {min_v}")
    print(f"-> Максимальний індекс VHI: {max_v}")
else:
    print(f"Даних для Області ID {target_area} за {target_year} рік не знайдено.")

РЕЗУЛЬТАТ ВИКОНАННЯ ЗАВДАННЯ 1:
Область ID: 9, Рік: 2020
-> Мінімальний індекс VHI: 26.22
-> Максимальний індекс VHI: 64.77


### Завдання 2. Знайти роки, в які спостерігалися сильні та екстремальні посухи (VHI менше заданого порогу) для обраної області.

In [5]:
def find_drought_years(dataframe, area_id, vhi_threshold=20.0):
    """Повертає список унікальних років, коли VHI падав нижче критичного порогу"""
    filtered = dataframe[(dataframe['Area_ID'] == area_id) & (dataframe['VHI'] < vhi_threshold)]
    unique_years = sorted(filtered['Year'].unique())
    return unique_years

In [6]:
# Шукаємо роки з екстремальною посухою (VHI < 15) для Області №12 (Львівська)
area_test = 12
threshold_test = 15.0

drought_years = find_drought_years(df, area_test, threshold_test)

print(f"РЕЗУЛЬТАТ ВИКОНАННЯ ЗАВДАННЯ 2:")
print(f"Роки з екстремальною посухою (VHI < {threshold_test}) для області ID {area_test}:")
print(drought_years)

РЕЗУЛЬТАТ ВИКОНАННЯ ЗАВДАННЯ 2:
Роки з екстремальною посухою (VHI < 15.0) для області ID 12:
[np.int64(1984), np.int64(1985), np.int64(1994), np.int64(1995), np.int64(2000), np.int64(2003), np.int64(2004), np.int64(2005)]


### Завдання 3. Фільтрація всього датасету за умовою: вивести роки та тижні, де індекс VHI знаходився в межах помірної посухи (від 20 до 35).

In [7]:
def filter_moderate_drought(dataframe):
    """Фільтрує записи, де VHI знаходиться в межах від 20 до 35 включно"""
    result_df = dataframe[dataframe['VHI'].between(20.0, 35.0)]
    return result_df[['Year', 'Week', 'VHI', 'Area_ID']]

In [8]:
# Отримуємо відфільтровану таблицю
moderate_drought_data = filter_moderate_drought(df)

print(f"РЕЗУЛЬТАТ ВИКОНАННЯ ЗАВДАННЯ 3:")
print(f"Всього знайдено записів з помірною посухою: {len(moderate_drought_data)}")
# Показуємо перші 10 рядків результату у вигляді красивої таблиці Pandas
moderate_drought_data.head(10)

РЕЗУЛЬТАТ ВИКОНАННЯ ЗАВДАННЯ 3:
Всього знайдено записів з помірною посухою: 7775


,Year,Week,VHI,Area_ID
4,1982,6,34.91,1
5,1982,7,33.14,1
6,1982,8,32.72,1
7,1982,9,32.77,1
8,1982,10,32.23,1
9,1982,11,30.38,1
10,1982,12,31.12,1
11,1982,13,31.65,1
12,1982,14,32.61,1
38,1982,40,34.87,1
